# databricks_launchpad_widgets\nGenerated from 00_common/databricks_launchpad_widgets.py for Databricks notebook import.\n

In [ ]:
"""Databricks UI widget helper driven by config/widget_options.yaml."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any\n\nimport yaml\n\n# ---------------------------------------------------------------------------\n# dbutils compatibility shim\n# ---------------------------------------------------------------------------\n# In a live Databricks notebook, `dbutils` is injected as a notebook global.\n# Outside Databricks (local dev, unit tests) it doesn't exist, so we provide\n# a no-op stub so this module can always be imported and tested safely.\n# ---------------------------------------------------------------------------\n\ntry:\n    dbutils  # type: ignore[name-defined]  # noqa: F821\nexcept NameError:\n\n    class _NoOpWidgets:  # pragma: no cover\n        """Stub that silently ignores widget creation outside Databricks."""\n\n        _values: dict[str, str] = {}\n\n        def dropdown(self, name: str, default: str, choices: list[str], label: str = "") -> None:\n            self._values[name] = default\n\n        def get(self, name: str) -> str:\n            return self._values.get(name, "")\n\n        def removeAll(self) -> None:\n            self._values.clear()\n\n    class _NoOpDbutils:\n        widgets = _NoOpWidgets()\n\n    dbutils = _NoOpDbutils()  # type: ignore[assignment]\n\n\n# ---------------------------------------------------------------------------\n# YAML helpers\n# ---------------------------------------------------------------------------\n\n\ndef load_widget_options(widget_file: str) -> dict:\n    path = Path(widget_file)\n    if not path.exists():\n        raise FileNotFoundError(f"Widget options file not found: {path}")\n    with open(path, "r", encoding="utf-8") as f:\n        payload = yaml.safe_load(f) or {}\n    return payload.get("widget_options", {})\n\n\ndef _choices(options: dict, key: str) -> list[str]:\n    values = options.get(key, [])\n    return [str(v) for v in values] if isinstance(values, list) else []\n\n\ndef _first(choices: list[str], fallback: str = "") -> str:\n    """Return the first choice as the dropdown default, or *fallback* if empty."""\n    return choices[0] if choices else fallback\n\n\n# ---------------------------------------------------------------------------\n# Widget creation\n# ---------------------------------------------------------------------------\n\n\ndef create_launchpad_widgets(widget_file: str | None = None) -> dict[str, Any]:\n    """Create Databricks dropdown widgets driven by widget_options.yaml.\n\n    Defaults for each widget are taken from the **first value** in the\n    corresponding YAML list, so updating widget_options.yaml automatically\n    refreshes the defaults without touching this file.\n\n    Returns the parsed ``widget_options`` dict so callers can inspect choices.\n    Calling this function more than once is safe — existing widgets are removed\n    first to avoid Databricks duplicate-widget errors.\n    """\n    if widget_file is None:\n        widget_file = str(Path(__file__).resolve().parents[2] / "config" / "widget_options.yaml")\n\n    options = load_widget_options(widget_file)\n\n    # Remove any previously registered widgets to prevent duplicate errors\n    # when the notebook cell is re-run.\n    try:\n        dbutils.widgets.removeAll()\n    except Exception:  # noqa: BLE001\n        pass\n\n    env_choices = _choices(options, "environments")\n    bool_choices = _choices(options, "boolean_values")\n    folder_choices = _choices(options, "metadata_folder_name")\n    mount_choices = _choices(options, "metadata_base_mount_point")\n    paths_choices = _choices(options, "paths_config_path")\n    source_choices = _choices(options, "ikea_dropzone_mounts")\n\n    dbutils.widgets.dropdown("environment", _first(env_choices, "dev"), env_choices, "Environment")\n    dbutils.widgets.dropdown("full_load", _first(bool_choices, "False"), bool_choices, "Full Load")\n    dbutils.widgets.dropdown("metadata_folder", _first(folder_choices), folder_choices, "Metadata Folder")\n    dbutils.widgets.dropdown("metadata_mount", _first(mount_choices), mount_choices, "Metadata Base Mount")\n    dbutils.widgets.dropdown("paths_config_path", _first(paths_choices), paths_choices, "Paths Config")\n    dbutils.widgets.dropdown("source_mount", _first(source_choices), source_choices, "Source Mount")\n\n    return options\n\n\n# ---------------------------------------------------------------------------\n# Widget value reader\n# ---------------------------------------------------------------------------\n\n_WIDGET_NAMES = [\n    "environment",\n    "full_load",\n    "metadata_folder",\n    "metadata_mount",\n    "paths_config_path",\n    "source_mount",\n]\n\n\ndef get_widget_values() -> dict[str, str]:\n    """Read all launchpad widget values and return them as a plain dict.\n\n    Call this after ``create_launchpad_widgets()`` to get the user-selected\n    values at notebook runtime.\n\n    Example::\n\n        create_launchpad_widgets()\n        cfg = get_widget_values()\n        env = cfg["environment"]   # "dev" | "qa" | "prod"\n    """\n    return {name: dbutils.widgets.get(name) for name in _WIDGET_NAMES}\n\n